# Ray Train 分布式微调练习

本 Notebook 面向大三数据科学专业学生，帮助你练习 **Ray Train** 分布式训练的核心用法。

## 学习目标
- 理解 Ray Train 的 `TorchTrainer` 工作原理
- 掌握 `ScalingConfig`、`RunConfig` 的配置方法
- 学会在训练函数中使用 `train.get_context()` 获取 Worker 信息
- 学会使用 `train.torch.prepare_model()` 和 `train.torch.prepare_data_loader()` 进行分布式包装
- 学会使用 `train.report()` 上报训练指标

## 任务
请搜索所有 `# TODO:` 注释，填写缺失的 Ray Train 核心代码，使训练能够正常运行。


In [ ]:
# ========== 环境检查 & 依赖安装 ==========
!pip install -q ray[train] torch transformers datasets modelscope

In [ ]:
# ========== 国内镜像加速 ==========
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
print("HuggingFace 镜像已设置: https://hf-mirror.com")

In [ ]:
import os

import ray
import torch
from datasets import load_dataset
from ray import train
from ray.train import RunConfig, ScalingConfig, CheckpointConfig
from ray.train.torch import TorchTrainer
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    get_linear_schedule_with_warmup,
)

# ========== 从 ModelScope 下载模型（国内高校网可访问）==========
_HF_MODEL_ID = "Qwen/Qwen3.5-0.8B"
_CACHE_DIR = "/data/qwen_model"

from modelscope import snapshot_download

_model_local = os.path.normpath(os.path.join(_CACHE_DIR, _HF_MODEL_ID))
if not os.path.isdir(_model_local):
    print(f"Downloading {_HF_MODEL_ID} from ModelScope (first time, ~3GB)...")
    _model_local = os.path.normpath(snapshot_download(_HF_MODEL_ID, cache_dir=_CACHE_DIR))

MODEL_PATH = _model_local
print(f"MODEL_PATH: {MODEL_PATH}")

# ========== 数据路径 ==========
DATASET_PATH = "/data/train_2k_qwen.jsonl"
VAL_DATASET_PATH = "/data/val_500_qwen.jsonl"

print(f"DATASET_PATH: {DATASET_PATH}")

## 1. 数据预处理函数

将 DataMind-12K 的 trajectory 字段转换为训练序列。已适配 Qwen3 的 chat template 格式。

In [ ]:
def preprocess_dataset(dataset, tokenizer, max_length: int = 2048):
    """
    Tokenize conversations — labels only assistant content.
    Uses offset mapping on apply_chat_template output.
    """

    def tokenize_fn(examples: dict) -> dict:
        input_ids_list = []
        labels_list = []
        skipped = 0

        for trajectory in examples["trajectory"]:
            messages = [{"role": m["role"], "content": m["content"]}
                        for m in trajectory]

            full_text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False,
            )

            encoding = tokenizer(
                full_text, return_offsets_mapping=True, add_special_tokens=False,
            )
            input_ids = encoding["input_ids"]
            offsets = encoding["offset_mapping"]
            labels = [-100] * len(input_ids)

            asst_found = False
            search_start = 0
            while True:
                start = full_text.find("<|im_start|>assistant\n", search_start)
                if start == -1:
                    break

                # End = next <|im_start|> or end of text
                next_start = full_text.find("<|im_start|>", start + 1)
                end = next_start if next_start != -1 else len(full_text)

                # Map character positions to token indices via offset mapping
                start_tok = None
                end_tok = None
                for i, (ch_s, ch_e) in enumerate(offsets):
                    if start_tok is None and ch_s <= start < ch_e:
                        start_tok = i
                    if ch_s < end <= ch_e:
                        end_tok = i
                        break
                if start_tok is None:
                    start_tok = 0
                if end_tok is None:
                    end_tok = len(offsets) - 1

                for i in range(start_tok, end_tok + 1):
                    labels[i] = input_ids[i]
                asst_found = True
                search_start = end

            if not asst_found:
                skipped += 1
                continue

            if len(input_ids) > max_length:
                input_ids = input_ids[:max_length]
                labels = labels[:max_length]

            input_ids_list.append(input_ids)
            labels_list.append(labels)

        if skipped:
            print(f"  [WARN] Skipped {skipped} samples (no assistant tokens)")

        return {"input_ids": input_ids_list, "labels": labels_list}

    return dataset.map(
        tokenize_fn, batched=True,
        remove_columns=dataset.column_names,
        desc="Tokenizing trajectories",
    )

In [ ]:
# ========== 预 Tokenize ==========
print("正在加载 Tokenizer...")
_pre_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, padding_side="right")
if _pre_tokenizer.pad_token is None:
    _pre_tokenizer.pad_token = _pre_tokenizer.eos_token

print(f"正在加载原始数据: {DATASET_PATH}")
_raw_dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
_raw_dataset = _raw_dataset.select(range(min(2000, len(_raw_dataset))))
print(f"共 {len(_raw_dataset)} 条样本")

print("正在 Tokenize（max_length=1024）...")
_tokenized = preprocess_dataset(_raw_dataset, _pre_tokenizer, max_length=1024)

TOKENIZED_TRAIN_PATH = "/data/tokenized_train"
_tokenized.save_to_disk(TOKENIZED_TRAIN_PATH)
print(f"预 tokenize 完成，已保存到 {TOKENIZED_TRAIN_PATH}")

del _pre_tokenizer, _raw_dataset, _tokenized
print("内存已释放。")

## 2. 训练函数

下面是 Ray Train 分布式训练函数，所有 `# TODO` 已补全。

In [ ]:
def train_func(config: dict):
    """
    全量微调（L20 48GB 优化版）。
    """
    from datasets import Dataset
    from ray.train import Checkpoint
    import tempfile
    import traceback

    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    try:
        model_path = config["model_path"]
        tokenized_data_path = config["tokenized_data_path"]
        epochs = config.get("epochs", 1)
        batch_size = config.get("batch_size", 4)
        learning_rate = config.get("lr", 5e-5)
        weight_decay = config.get("weight_decay", 0.01)
        max_length = config.get("max_length", 1024)
        grad_accum = config.get("gradient_accumulation_steps", 4)

        context = train.get_context()
        worker_rank = context.get_world_rank()
        world_size = context.get_world_size()

        print(f"[Worker {worker_rank}/{world_size}] batch_size={batch_size}, grad_accum={grad_accum}")

        tokenizer = AutoTokenizer.from_pretrained(model_path, padding_side="right")
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16)
        model.config.pad_token_id = tokenizer.pad_token_id
        model.gradient_checkpointing_enable()

        tokenized_dataset = Dataset.load_from_disk(tokenized_data_path)
        if worker_rank == 0:
            print(f"数据集: {len(tokenized_dataset)} 条样本")

        collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, return_tensors="pt")
        dataloader = DataLoader(tokenized_dataset, batch_size=batch_size, shuffle=True, collate_fn=collator)
        dataloader = train.torch.prepare_data_loader(dataloader)
        model = train.torch.prepare_model(model)

        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        total_steps = (len(tokenized_dataset) // (batch_size * world_size)) * epochs
        warmup_steps = max(1, int(total_steps * 0.05))
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

        model.train()
        global_step = 0
        print(f"[Worker {worker_rank}] {len(dataloader)} batches, {total_steps} steps")

        for epoch in range(epochs):
            total_loss = 0.0
            num_batches = 0
            for batch in dataloader:
                outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["labels"])
                loss = outputs.loss / grad_accum
                loss.backward()
                num_batches += 1
                if (num_batches % grad_accum == 0) or (num_batches == len(dataloader)):
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                    global_step += 1
                total_loss += loss.item() * grad_accum

                if num_batches <= 3 or num_batches % 50 == 0:
                    print(f"[Worker {worker_rank}] {num_batches}/{len(dataloader)}, loss={loss.item() * grad_accum:.4f}")

            avg_loss = total_loss / num_batches
            print(f"[Worker {worker_rank}] Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}")

            with tempfile.TemporaryDirectory() as tmpdir:
                state_dict = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
                torch.save(state_dict, os.path.join(tmpdir, "model.pt"))
                tokenizer.save_pretrained(tmpdir)
                train.report(
                    metrics={"epoch": epoch, "loss": avg_loss, "learning_rate": scheduler.get_last_lr()[0], "global_step": global_step},
                    checkpoint=Checkpoint.from_directory(tmpdir),
                )

        print(f"[Worker {worker_rank}] 训练完成！")
    except Exception as e:
        print(f"[Worker ERROR] {e}")
        traceback.print_exc()
        raise

## 3. 启动 Ray Train 分布式训练

下方代码配置 `ScalingConfig`、`RunConfig` 和 `TorchTrainer`，启动分布式训练。

In [ ]:
def main():
    if not ray.is_initialized():
        ray.init(ignore_reinit_error=True)
    print(f"GPU 可用: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU 型号: {torch.cuda.get_device_name(0)}")
        print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.0f} GB")

    train_config = {
        "model_path": MODEL_PATH,
        "tokenized_data_path": TOKENIZED_TRAIN_PATH,
        "epochs": 1,
        "batch_size": 4,
        "lr": 5e-5,
        "weight_decay": 0.01,
        "max_length": 1024,
        "gradient_accumulation_steps": 4,
    }

    use_gpu = torch.cuda.is_available()
    scaling_config = ScalingConfig(num_workers=1, use_gpu=use_gpu)

    storage_path = "/data/ray_results"
    os.makedirs(storage_path, exist_ok=True)
    run_config = RunConfig(
        name="qwen-datamind-v2",
        storage_path=storage_path,
        checkpoint_config=CheckpointConfig(
            num_to_keep=1,
            checkpoint_score_attribute="loss",
            checkpoint_score_order="min",
        ),
    )

    trainer = TorchTrainer(
        train_loop_per_worker=train_func,
        train_loop_config=train_config,
        scaling_config=scaling_config,
        run_config=run_config,
    )
    result = trainer.fit()

    print("\n" + "=" * 60)
    print("微调完成！")
    print("=" * 60)
    print(f"实验路径:   {result.path}")
    print(f"最终指标:   {result.metrics}")
    print(f"Checkpoint:  {result.checkpoint}")
    print("=" * 60)

    if ray.is_initialized():
        ray.shutdown()
    print("\nRay 已关闭。")

In [ ]:
# ========== 运行训练 ==========
main()